In [1]:
import sys
from google import genai
sys.path.append('../')
from src.load_data import *
from src.agents import *
from src.fine_tuned_agents import *

In [2]:
testing_data = load_data_from_csv('../data/test_sent_emo.csv')
print(testing_data.head())

Data loaded successfully from ../data/test_sent_emo.csv
   Sr No.                                          Utterance Speaker  \
0       1  Why do all youre coffee mugs have numbers on ...    Mark   
1       2  Oh. Thats so Monica can keep track. That way ...  Rachel   
2       3                                       Y'know what?  Rachel   
3      19                     Come on, Lydia, you can do it.    Joey   
4      20                                              Push!    Joey   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  surprise  positive            0             0       3       19   
1     anger  negative            0             1       3       19   
2   neutral   neutral            0             2       3       19   
3   neutral   neutral            1             0       1       23   
4       joy  positive            1             1       1       23   

      StartTime       EndTime  
0  00:14:38,127  00:14:40,378  
1  00:14:40,629  00:14:47,385  


In [3]:
testing_data.drop_duplicates(inplace=True)

In [4]:
# Create a unique key like "102_5" (Dialogue_ID + Utterance_ID)
testing_data['Recognition_ID'] = (
    testing_data['Dialogue_ID'].astype(str) + "_" + 
    testing_data['Utterance_ID'].astype(str) + "_" +
    testing_data['Season'].astype(str) + "_" +
    testing_data['Episode'].astype(str)
)

print(testing_data[['Dialogue_ID', 'Utterance_ID', 'Recognition_ID']].head())

   Dialogue_ID  Utterance_ID Recognition_ID
0            0             0       0_0_3_19
1            0             1       0_1_3_19
2            0             2       0_2_3_19
3            1             0       1_0_1_23
4            1             1       1_1_1_23


In [5]:
def preprocess_test_data(testing_data):
    df = testing_data.copy()
    # Ensure chronological order
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])
    
    scenes = []
    for diag_id, group in df.groupby('Dialogue_ID'):
        # 1. The 'Public' data we give to agents (No labels!)
        agent_view = group[['Utterance', 'Speaker', 'Recognition_ID']].to_dict(orient='records')
        
        # 2. The 'Private' data for F1 calculation
        ground_truth = group[['Recognition_ID', 'Emotion', 'Sentiment']].to_dict(orient='records')
        
        scene_data = {
            "dialogue_id": diag_id,
            "utterances": agent_view,  # Sent to Agents
            "ground_truth": ground_truth, # Saved for Evaluation
            "total_turns": len(group)
        }
        scenes.append(scene_data)
    return scenes

preprocessed_test_data = preprocess_test_data(testing_data)
preprocessed_test_data_df = pd.DataFrame(preprocessed_test_data)
print(preprocessed_test_data_df.head())
print(preprocessed_test_data[0])
# Check exactly what the agents are seeing
print(f"Available keys in your data: {preprocessed_test_data[0]['utterances'][0].keys()}")

   dialogue_id                                         utterances  \
0            0  [{'Utterance': 'Why do all youre coffee mugs ...   
1            1  [{'Utterance': 'Come on, Lydia, you can do it....   
2            2  [{'Utterance': 'Okay.', 'Speaker': 'Ross', 'Re...   
3            3  [{'Utterance': 'Oh, okay, I get it.', 'Speaker...   
4            4  [{'Utterance': 'Ohh!', 'Speaker': 'Phoebe', 'R...   

                                        ground_truth  total_turns  
0  [{'Recognition_ID': '0_0_3_19', 'Emotion': 'su...            3  
1  [{'Recognition_ID': '1_0_1_23', 'Emotion': 'ne...            8  
2  [{'Recognition_ID': '2_0_5_16', 'Emotion': 'ne...           11  
3  [{'Recognition_ID': '3_0_5_15', 'Emotion': 'ne...            7  
4  [{'Recognition_ID': '4_0_4_14', 'Emotion': 'su...            9  
{'dialogue_id': 0, 'utterances': [{'Utterance': 'Why do all you\x92re coffee mugs have numbers on the bottom?', 'Speaker': 'Mark', 'Recognition_ID': '0_0_3_19'}, {'Utterance': '

In [6]:
def run_phase2_council(scene_obj):
    """
    Processes a single scene using the keys present in your data.
    """
    dialogue_id = scene_obj["dialogue_id"]
    utterances = scene_obj["utterances"]
    ground_truth_map = {
        item['Recognition_ID']: {
            'Emotion': item['Emotion'], 
            'Sentiment': item['Sentiment']
        } 
        for item in scene_obj.get("ground_truth", [])
    }

    # Format script for Level 1 agents
    scene_script = "\n".join([f"{u['Speaker']}: {u['Utterance']}" for u in utterances])

    # -- LEVEL 1: GLOBAL CONTEXT --
    global_context = groq_llm_call(
        prompt=f"{context_manager_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        api_key=context_manager_api
    )
    social_graph = groq_llm_call(
        prompt=f"{relational_graph_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        api_key=relational_graph_api
    )

    scene_results = []

    # -- LEVEL 2 & 3: UTTERANCE PROCESSING --
    for u in utterances:
        # Use .get() to avoid KeyErrors if a row is malformed
        target_text = u.get('Utterance', "")
        speaker = u.get('Speaker', "Unknown")
        rec_id = u.get('Recognition_ID', "unknown_id") # Critical for F1 mapping
        actual_labels = ground_truth_map.get(rec_id, {"Emotion": "neutral", "Sentiment": "neutral"})
        actual_emotion = actual_labels["Emotion"]
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            f3 = executor.submit(call_tuned_profiler, target_text, social_graph)
            f4 = executor.submit(call_tuned_sentiment, target_text)
            
            profile_report = f3.result()
            sentiment_report = f4.result()

        dynamics_report = call_social_dynamics(target_text, profile_report, social_graph)

        # Agent 6: Final Aggregator
        # We pass rec_id so it can be preserved in the JSON output
        final_prediction_raw = call_gpt_oss_aggregator(
            rec_id, target_text, global_context, profile_report, sentiment_report, dynamics_report
        )

        scene_results.append({
            "Dialogue_ID": dialogue_id,
            "Recognition_ID": rec_id,
            "Speaker": speaker,
            "Utterance": target_text,
            "Predicted_Emotion_Raw": final_prediction_raw,
            "Actual_Emotion": actual_emotion # Ground truth check
        })
    
    return scene_results

In [7]:
import os
import pandas as pd

output_file = "../logs/council_phase2_results.csv"

# 1. Identify which scenes are already done
processed_ids = set()
if os.path.exists(output_file):
    try:
        existing_df = pd.read_csv(output_file)
        if not existing_df.empty:
            processed_ids = set(existing_df['Dialogue_ID'].unique())
            print(f"🔄 Found {len(processed_ids)} scenes already processed. Skipping them...")
    except Exception as e:
        print(f"⚠️ Could not read existing results: {e}")

all_final_results = []

# 2. Start the loop
for i, scene in enumerate(preprocessed_test_data):
    scene_id = scene['dialogue_id']
    
    # Check if we should skip
    if scene_id in processed_ids:
        continue
    
    print(f"🎬 Processing Scene {i+1}/{len(preprocessed_test_data)} (ID: {scene_id})...")
    
    try:
        scene_out = run_phase2_council(scene)
        
        # Append to a list for current session
        all_final_results.extend(scene_out)
        
        # 3. SAVE IMMEDIATELY (Append Mode)
        # We convert only the NEW scene to a dataframe
        new_df = pd.DataFrame(scene_out)
        
        # Write to CSV: if file doesn't exist, write header; else append without header
        new_df.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
        
        print(f"💾 Scene {scene_id} saved to {output_file}")

    except Exception as e:
        error_msg = str(e).lower()
        if "429" in error_msg or "resource exhausted" in error_msg:
            print(f"🛑 Rate limit exhausted at Scene {scene_id}. Stop and wait before re-running.")
            print(e)
            break # Exit the loop so you can manually restart later
        else:
            print(f"⚠️ Error in Scene {scene_id}: {e}")
            continue # Move to the next one if it's a non-quota error

print("✅ SESSION FINISHED. All unprocessed scenes in this batch are done!")

🔄 Found 10 scenes already processed. Skipping them...
🎬 Processing Scene 11/280 (ID: 10)...
💾 Scene 10 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 12/280 (ID: 11)...
💾 Scene 11 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 13/280 (ID: 12)...
💾 Scene 12 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 14/280 (ID: 13)...
💾 Scene 13 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 15/280 (ID: 14)...
💾 Scene 14 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 16/280 (ID: 15)...
💾 Scene 15 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 17/280 (ID: 16)...
💾 Scene 16 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 18/280 (ID: 17)...
💾 Scene 17 saved to ../logs/council_phase2_results.csv
🎬 Processing Scene 19/280 (ID: 18)...
🛑 Rate limit exhausted at Scene 18. Stop and wait before re-running.
429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex

In [9]:
import pandas as pd
import json
import re
from sklearn.metrics import classification_report, f1_score, confusion_matrix

def generate_erc_report(csv_path):
    # 1. Load the results
    df = pd.read_csv(csv_path)
    
    def extract_label(raw_string):
        """Extracts 'predicted_emotion' from the Aggregator's JSON string."""
        if pd.isna(raw_string):
            return "neutral"
        try:
            # Handle potential double-quoting from CSV exports
            clean_json = raw_string.replace('""', '"')
            # Extract content between curly braces if extra text exists
            match = re.search(r'\{.*\}', clean_json, re.DOTALL)
            if match:
                data = json.loads(match.group())
                return data.get("predicted_emotion", "neutral").lower().strip()
            return "neutral"
        except Exception:
            return "neutral"

    # 2. Clean and Align Labels
    df['Predicted_Clean'] = df['Predicted_Emotion_Raw'].apply(extract_label)
    df['Actual_Clean'] = df['Actual_Emotion'].str.lower().str.strip()

    # 3. Define the labels to include (Excluding 'neutral' if you want Weighted F1)
    # MELD labels: neutral, joy, surprise, anger, sadness, disgust, fear
    labels = sorted(df['Actual_Clean'].unique())

    # 4. Generate the Scikit-Learn Report
    print("\n" + "="*60)
    print("      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT")
    print("="*60)
    
    report = classification_report(
        df['Actual_Clean'], 
        df['Predicted_Clean'], 
        labels=labels,
        digits=4
    )
    print(report)

    # 5. Calculate Weighted F1 (The main metric for MELD research)
    wf1 = f1_score(df['Actual_Clean'], df['Predicted_Clean'], average='weighted')
    print(f"\nOVERALL WEIGHTED F1 SCORE: {wf1:.4f}")
    print("="*60)

    return df

# Usage:
report_df = generate_erc_report("../logs/council_phase2_results.csv")


      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger     0.3256    0.9032    0.4786        31
     disgust     0.3333    0.1667    0.2222         6
        fear     0.6000    0.5000    0.5455         6
         joy     0.4545    0.5769    0.5085        26
     neutral     0.8372    0.4235    0.5625        85
     sadness     0.0000    0.0000    0.0000        12
    surprise     0.6250    0.3333    0.4348        15

    accuracy                         0.4862       181
   macro avg     0.4537    0.4148    0.3932       181
weighted avg     0.5970    0.4862    0.4807       181


OVERALL WEIGHTED F1 SCORE: 0.4807
